In [23]:
# from train.py
import numpy as np
from omegaconf import OmegaConf
import pathlib

from torch import optim

from models import *
from tasks import *
import config
import utils

import torch

config_arg = "spatial_navigation/noisy_unbiased"

In [28]:
# from config.py
#    """Loads the experiment configuration for model training."""
#    cli_config = OmegaConf.from_cli()

train_config_path = pathlib.Path("configs", "train")
base_config = OmegaConf.load(train_config_path / "base.yml")
task_config = OmegaConf.load(train_config_path / config_arg.split('/')[0] / "base.yml")
expt_config = OmegaConf.load(train_config_path / f"{config_arg}.yml")
expt_config.device = "cuda" if torch.cuda.is_available() else "cpu"
args = OmegaConf.merge(base_config, task_config, expt_config)
args['config'] = config_arg
args['seed'] = 0
args['rnn']['bias'] = True
args['trainer']['n_epochs'] = 10_000

In [29]:
# from train.py
task = spatial_navigation.SpatialNavigation(
    box_width=args.task.box_width,
    box_height=args.task.box_height,
    border_region=args.task.border_region,
    border_slow_factor=args.task.border_slow_factor,
    init_pos=args.task.init_pos,
    biased=args.task.biased,
    drift_const=args.task.drift_const,
    anchor_point=np.array(args.task.anchor_point),
    dt=args.task.dt,
    mu=args.task.mu,
    sigma=args.task.sigma,
    b=args.task.b,
    use_place_cells=args.task.use_place_cells,
    place_cells_num=args.task.place_cells_num,
    place_cells_sigma=args.task.place_cells_sigma,
    place_cells_dog=args.task.place_cells_dog,
    place_cells_surround_scale=args.task.place_cells_surround_scale,
    sequence_length=args.task.sequence_length,
    batch_size=args.task.batch_size,
    device=args.device,
)

model = rnn.RNN(
    task=task,
    n_in=args.rnn.n_in,
    n_rec=args.rnn.n_rec,
    n_out=args.rnn.n_out,
    n_init=args.rnn.n_init,
    sigma_in=np.sqrt(args.rnn.sigma2_in),
    sigma_rec=np.sqrt(args.rnn.sigma2_rec),
    sigma_out=np.sqrt(args.rnn.sigma2_out),
    dt=args.rnn.dt,
    tau=args.rnn.tau,
    bias=args.rnn.bias,
    activation_fn=args.rnn.activation_fn,
    device=args.device,
)

In [30]:
args.rnn

{'model': 'vanilla', 'n_in': 2, 'n_rec': 512, 'n_out': 512, 'n_init': 512, 'sigma2_in': 0, 'sigma2_rec': 0.0001, 'sigma2_out': 0, 'dt': 0.2, 'tau': 1, 'bias': True, 'activation_fn': 'relu'}

In [24]:
# from trainer.py __init
train_data = task.get_generator()
test_data = task.get_test_batch()
unmask_every = 6
lr=0.001
weight_decay=0
compute_all_metrics=True
test_freq=100
save_freq=100
path="results"
device="cuda"

model.set_device(device) #self.device)
optimizer = optim.Adam(
    model.parameters(), lr=lr, weight_decay=weight_decay
)


# from trainer.py train
n_epochs = args.trainer.n_epochs
start_epoch = 0

epoch = start_epoch
train_metrics = dict() #self.train_metrics = dict()
test_metrics = dict() #self.test_metrics = dict()
aux = None
unmask_levels = torch.arange(unmask_every) + 1 #self.unmask_every) + 1 #[1], [1,2], [1,2,3], etc. masking code.

for batch in train_data: #self.train_data:
    # each unmasking level gets the same amount of training time
    #unmask_level = unmask_levels[min(self.unmask_every - 1, \
    #                             int(epoch // (n_epochs / self.unmask_every)))] # masking code
    unmask_level = unmask_levels[min(unmask_every - 1, \
                                 int(epoch // (n_epochs / unmask_every)))] # masking code    
    epoch += 1
    if epoch - start_epoch > n_epochs:
        break

    model.train() #self.model.train()
    optimizer.zero_grad() #self.optimizer.zero_grad()

    data = batch["data"].to(device) #self.device)
    data[:, torch.arange(data.shape[1]) % unmask_level != 0] = 0  # masking code
    init_state = batch["init_state"].to(device) #self.device)
    targets = batch["targets"].to(device) #self.device)

    #_, outputs = self.model(data, init_state=init_state)
    _, outputs = model(data, init_state=init_state)
    
    if compute_all_metrics: #self.compute_all_metrics:
        aux = batch

    #train_loss, train_metric = self.model.task.compute_metrics(outputs, targets, aux)
    train_loss, train_metric = model.task.compute_metrics(outputs, targets, aux)
    train_loss.backward()
    optimizer.step() #self.optimizer.step()
    self.train_metrics[epoch] = train_metric.copy() #self.train_metrics[epoch] = train_metric.copy()

    print(f"Epoch {epoch} (train):")
    for k, v in train_metric.items():
        print(f"  - {k} = {v}.")

    #if epoch % self.save_freq == 0:
    #    model_path = pathlib.Path(self.path).joinpath(f"model_{epoch}.pt")
    #    torch.save(self.model, model_path)
    #    print(f"Model saved at epoch {epoch}.")

    if epoch % test_freq == 0: #self.test_freq == 0:
        with torch.no_grad():
            model.eval() #self.model.eval()

            #data = self.test_data["data"].to(self.device)
            #init_state = self.test_data["init_state"].to(self.device)
            #targets = self.test_data["targets"].to(self.device)
            data = test_data["data"].to(device)
            init_state = test_data["init_state"].to(device)
            targets = test_data["targets"].to(device)

            if compute_all_metrics: #self.compute_all_metrics:
                aux = test_data #self.test_data

            #_, outputs = self.model(data, init_state=init_state)
            _, outputs = model(data, init_state=init_state)
            
            #_, test_metric = self.model.task.compute_metrics(outputs, targets, aux)
            #self.test_metrics[epoch] = test_metric.copy()
            _, test_metric = model.task.compute_metrics(outputs, targets, aux)
            test_metrics[epoch] = test_metric.copy()            

            print(f"Epoch {epoch} (test):")
            for k, v in test_metric.items():
                print(f"  - {k} = {v}.")

#train_metrics_path = pathlib.Path(self.path).joinpath("train_metrics.json")
#json.dump(self.train_metrics, open(train_metrics_path, "w"), indent=4)

#test_metrics_path = pathlib.Path(self.path).joinpath("test_metrics.json")
#json.dump(self.test_metrics, open(test_metrics_path, "w"), indent=4)

Epoch 1 (train):
  - loss = 0.03356403857469559.
  - pos_mse = 0.6224098205566406.
Epoch 2 (train):
  - loss = 0.008595512248575687.
  - pos_mse = 0.721616268157959.
Epoch 3 (train):
  - loss = 0.0036734328605234623.
  - pos_mse = 0.5310875177383423.
Epoch 4 (train):
  - loss = 0.0034624766558408737.
  - pos_mse = 0.5501541495323181.
Epoch 5 (train):
  - loss = 0.003700995584949851.
  - pos_mse = 0.929489254951477.
Epoch 6 (train):
  - loss = 0.003682002890855074.
  - pos_mse = 0.7592671513557434.
Epoch 7 (train):
  - loss = 0.003406862262636423.
  - pos_mse = 0.5784147381782532.
Epoch 8 (train):
  - loss = 0.0029840506613254547.
  - pos_mse = 0.476013720035553.
Epoch 9 (train):
  - loss = 0.0024955994449555874.
  - pos_mse = 0.5466363430023193.
Epoch 10 (train):
  - loss = 0.0020254580304026604.
  - pos_mse = 0.6151551008224487.
Epoch 11 (train):
  - loss = 0.0016060787020251155.
  - pos_mse = 0.5756192207336426.
Epoch 12 (train):
  - loss = 0.001273066271096468.
  - pos_mse = 0.54645

Epoch 97 (train):
  - loss = 6.082447725930251e-05.
  - pos_mse = 0.5640134215354919.
Epoch 98 (train):
  - loss = 6.13992742728442e-05.
  - pos_mse = 0.5848462581634521.
Epoch 99 (train):
  - loss = 6.231124280020595e-05.
  - pos_mse = 0.6054942011833191.
Epoch 100 (train):
  - loss = 6.074351040297188e-05.
  - pos_mse = 0.5812694430351257.
Epoch 100 (test):
  - loss = 6.120269245002419e-05.
  - pos_mse = 0.5820159912109375.
Epoch 101 (train):
  - loss = 6.142620259197429e-05.
  - pos_mse = 0.5767866969108582.
Epoch 102 (train):
  - loss = 6.203818338690326e-05.
  - pos_mse = 0.6039458513259888.
Epoch 103 (train):
  - loss = 6.136656156741083e-05.
  - pos_mse = 0.5897353887557983.
Epoch 104 (train):
  - loss = 6.073392432881519e-05.
  - pos_mse = 0.581501305103302.
Epoch 105 (train):
  - loss = 6.290763849392533e-05.
  - pos_mse = 0.633037269115448.
Epoch 106 (train):
  - loss = 6.140430923551321e-05.
  - pos_mse = 0.5938121676445007.
Epoch 107 (train):
  - loss = 5.9340731240808964e-

Epoch 192 (train):
  - loss = 5.8039579016622156e-05.
  - pos_mse = 0.5702189207077026.
Epoch 193 (train):
  - loss = 5.726061499444768e-05.
  - pos_mse = 0.5537429451942444.
Epoch 194 (train):
  - loss = 5.6253593356814235e-05.
  - pos_mse = 0.5504298806190491.
Epoch 195 (train):
  - loss = 5.731647979700938e-05.
  - pos_mse = 0.5669953227043152.
Epoch 196 (train):
  - loss = 5.720261106034741e-05.
  - pos_mse = 0.563167154788971.
Epoch 197 (train):
  - loss = 5.6634329666849226e-05.
  - pos_mse = 0.554650068283081.
Epoch 198 (train):
  - loss = 5.6453274737577885e-05.
  - pos_mse = 0.5497056245803833.
Epoch 199 (train):
  - loss = 5.6688229960855097e-05.
  - pos_mse = 0.5413779020309448.
Epoch 200 (train):
  - loss = 5.685914584319107e-05.
  - pos_mse = 0.5437509417533875.
Epoch 200 (test):
  - loss = 5.648983642458916e-05.
  - pos_mse = 0.5405333638191223.
Epoch 201 (train):
  - loss = 5.731651617679745e-05.
  - pos_mse = 0.5656948089599609.
Epoch 202 (train):
  - loss = 5.656818757

Epoch 285 (train):
  - loss = 5.596956907538697e-05.
  - pos_mse = 0.5492127537727356.
Epoch 286 (train):
  - loss = 5.5379838158842176e-05.
  - pos_mse = 0.535561740398407.
Epoch 287 (train):
  - loss = 5.509125912794843e-05.
  - pos_mse = 0.5487173795700073.
Epoch 288 (train):
  - loss = 5.534649972105399e-05.
  - pos_mse = 0.5240357518196106.
Epoch 289 (train):
  - loss = 5.577530828304589e-05.
  - pos_mse = 0.5382986664772034.
Epoch 290 (train):
  - loss = 5.456867802422494e-05.
  - pos_mse = 0.5175294876098633.
Epoch 291 (train):
  - loss = 5.51557241124101e-05.
  - pos_mse = 0.5349974036216736.
Epoch 292 (train):
  - loss = 5.404197872849181e-05.
  - pos_mse = 0.5215159058570862.
Epoch 293 (train):
  - loss = 5.515437078429386e-05.
  - pos_mse = 0.5226761102676392.
Epoch 294 (train):
  - loss = 5.466503353090957e-05.
  - pos_mse = 0.5203343629837036.
Epoch 295 (train):
  - loss = 5.555408279178664e-05.
  - pos_mse = 0.5301063060760498.
Epoch 296 (train):
  - loss = 5.598359712166

Epoch 380 (train):
  - loss = 5.333710214472376e-05.
  - pos_mse = 0.49619442224502563.
Epoch 381 (train):
  - loss = 5.4550455388380215e-05.
  - pos_mse = 0.522835910320282.
Epoch 382 (train):
  - loss = 5.3879623010288924e-05.
  - pos_mse = 0.5236796140670776.
Epoch 383 (train):
  - loss = 5.382253948482685e-05.
  - pos_mse = 0.5208554267883301.
Epoch 384 (train):
  - loss = 5.432699617813341e-05.
  - pos_mse = 0.5274895429611206.
Epoch 385 (train):
  - loss = 5.387015698943287e-05.
  - pos_mse = 0.517591118812561.
Epoch 386 (train):
  - loss = 5.531083297682926e-05.
  - pos_mse = 0.5579731464385986.
Epoch 387 (train):
  - loss = 5.2338036766741425e-05.
  - pos_mse = 0.4848003685474396.
Epoch 388 (train):
  - loss = 5.418070577434264e-05.
  - pos_mse = 0.5337772369384766.
Epoch 389 (train):
  - loss = 5.396602500695735e-05.
  - pos_mse = 0.5325424671173096.
Epoch 390 (train):
  - loss = 5.331755892257206e-05.
  - pos_mse = 0.5173420906066895.
Epoch 391 (train):
  - loss = 5.538320328

Epoch 475 (train):
  - loss = 5.397655695560388e-05.
  - pos_mse = 0.5254762172698975.
Epoch 476 (train):
  - loss = 5.334155503078364e-05.
  - pos_mse = 0.5114744305610657.
Epoch 477 (train):
  - loss = 5.407335629570298e-05.
  - pos_mse = 0.5249291658401489.
Epoch 478 (train):
  - loss = 5.4692307458026335e-05.
  - pos_mse = 0.5331491827964783.
Epoch 479 (train):
  - loss = 5.358806811273098e-05.
  - pos_mse = 0.5155826807022095.
Epoch 480 (train):
  - loss = 5.434744889498688e-05.
  - pos_mse = 0.5390624403953552.
Epoch 481 (train):
  - loss = 5.368826168705709e-05.
  - pos_mse = 0.5239448547363281.
Epoch 482 (train):
  - loss = 5.402256647357717e-05.
  - pos_mse = 0.5196430683135986.
Epoch 483 (train):
  - loss = 5.309590051183477e-05.
  - pos_mse = 0.5100854635238647.
Epoch 484 (train):
  - loss = 5.3263840527506545e-05.
  - pos_mse = 0.5190132856369019.
Epoch 485 (train):
  - loss = 5.466221409733407e-05.
  - pos_mse = 0.5359683036804199.
Epoch 486 (train):
  - loss = 5.357249756

... (17 outputs left out)

Epoch 2150 (train):
  - loss = 4.9653241148917004e-05.
  - pos_mse = 0.5537533760070801.
Epoch 2151 (train):
  - loss = 4.859722685068846e-05.
  - pos_mse = 0.48099055886268616.
Epoch 2152 (train):
  - loss = 4.978254219167866e-05.
  - pos_mse = 0.4749891459941864.
Epoch 2153 (train):
  - loss = 5.014956332161091e-05.
  - pos_mse = 0.4838706851005554.
Epoch 2154 (train):
  - loss = 4.942525993101299e-05.
  - pos_mse = 0.5302056670188904.
Epoch 2155 (train):
  - loss = 5.0515900511527434e-05.
  - pos_mse = 0.5332297682762146.
Epoch 2156 (train):
  - loss = 4.864603761234321e-05.
  - pos_mse = 0.4772651195526123.
Epoch 2157 (train):
  - loss = 5.1278624596307054e-05.
  - pos_mse = 0.5177316665649414.
Epoch 2158 (train):
  - loss = 4.9162412324221805e-05.
  - pos_mse = 0.49297070503234863.
Epoch 2159 (train):
  - loss = 4.9501744797453284e-05.
  - pos_mse = 0.5310551524162292.
Epoch 2160 (train):
  - loss = 4.969229121343233e-05.
  - pos_mse = 0.5005573034286499.
Epoch 2161 (train):
  - l

Epoch 2243 (train):
  - loss = 4.9419271817896515e-05.
  - pos_mse = 0.503034770488739.
Epoch 2244 (train):
  - loss = 4.919579077977687e-05.
  - pos_mse = 0.4854094982147217.
Epoch 2245 (train):
  - loss = 4.9372789362678304e-05.
  - pos_mse = 0.4908645749092102.
Epoch 2246 (train):
  - loss = 4.998019358026795e-05.
  - pos_mse = 0.4903799593448639.
Epoch 2247 (train):
  - loss = 4.883624205831438e-05.
  - pos_mse = 0.49517908692359924.
Epoch 2248 (train):
  - loss = 4.974834155291319e-05.
  - pos_mse = 0.48911526799201965.
Epoch 2249 (train):
  - loss = 4.871329656452872e-05.
  - pos_mse = 0.472612202167511.
Epoch 2250 (train):
  - loss = 4.927874033455737e-05.
  - pos_mse = 0.47701406478881836.
Epoch 2251 (train):
  - loss = 5.015859642298892e-05.
  - pos_mse = 0.5024952292442322.
Epoch 2252 (train):
  - loss = 4.874269143329002e-05.
  - pos_mse = 0.5227815508842468.
Epoch 2253 (train):
  - loss = 4.887060640612617e-05.
  - pos_mse = 0.49635449051856995.
Epoch 2254 (train):
  - loss

Epoch 2335 (train):
  - loss = 4.857557360082865e-05.
  - pos_mse = 0.47884178161621094.
Epoch 2336 (train):
  - loss = 4.7696419642306864e-05.
  - pos_mse = 0.46400994062423706.
Epoch 2337 (train):
  - loss = 5.054591747466475e-05.
  - pos_mse = 0.48098039627075195.
Epoch 2338 (train):
  - loss = 4.999644806957804e-05.
  - pos_mse = 0.5077847242355347.
Epoch 2339 (train):
  - loss = 4.854906728724018e-05.
  - pos_mse = 0.48801547288894653.
Epoch 2340 (train):
  - loss = 4.935435936204158e-05.
  - pos_mse = 0.4962979257106781.
Epoch 2341 (train):
  - loss = 4.7620127588743344e-05.
  - pos_mse = 0.4486851394176483.
Epoch 2342 (train):
  - loss = 4.936575351166539e-05.
  - pos_mse = 0.48686620593070984.
Epoch 2343 (train):
  - loss = 4.907533366349526e-05.
  - pos_mse = 0.4849902093410492.
Epoch 2344 (train):
  - loss = 4.8840662202565e-05.
  - pos_mse = 0.48294442892074585.
Epoch 2345 (train):
  - loss = 4.960405931342393e-05.
  - pos_mse = 0.5070501565933228.
Epoch 2346 (train):
  - lo

Epoch 2427 (train):
  - loss = 4.8641937610227615e-05.
  - pos_mse = 0.5126532912254333.
Epoch 2428 (train):
  - loss = 4.9396279791835696e-05.
  - pos_mse = 0.48958709836006165.
Epoch 2429 (train):
  - loss = 4.7995359636843204e-05.
  - pos_mse = 0.46661317348480225.
Epoch 2430 (train):
  - loss = 4.8405039706267416e-05.
  - pos_mse = 0.4633930027484894.
Epoch 2431 (train):
  - loss = 4.947282650391571e-05.
  - pos_mse = 0.488653302192688.
Epoch 2432 (train):
  - loss = 4.891338903689757e-05.
  - pos_mse = 0.4988609850406647.
Epoch 2433 (train):
  - loss = 4.8691796109778807e-05.
  - pos_mse = 0.5105670094490051.
Epoch 2434 (train):
  - loss = 4.771205203724094e-05.
  - pos_mse = 0.4764750003814697.
Epoch 2435 (train):
  - loss = 4.935299148201011e-05.
  - pos_mse = 0.46923813223838806.
Epoch 2436 (train):
  - loss = 4.9609971028985456e-05.
  - pos_mse = 0.48950594663619995.
Epoch 2437 (train):
  - loss = 4.87871075165458e-05.
  - pos_mse = 0.4766937494277954.
Epoch 2438 (train):
  - 

In [26]:
test_metrics

{100: {'loss': 6.120269245002419e-05, 'pos_mse': 0.5820159912109375},
 200: {'loss': 5.648983642458916e-05, 'pos_mse': 0.5405333638191223},
 300: {'loss': 5.509093898581341e-05, 'pos_mse': 0.5493630170822144},
 400: {'loss': 5.419374429038726e-05, 'pos_mse': 0.5155388116836548},
 500: {'loss': 5.35807412234135e-05, 'pos_mse': 0.5247265696525574},
 600: {'loss': 5.310981214279309e-05, 'pos_mse': 0.5189598202705383},
 700: {'loss': 5.2701310778502375e-05, 'pos_mse': 0.5170234441757202},
 800: {'loss': 5.231895192991942e-05, 'pos_mse': 0.519228458404541},
 900: {'loss': 5.188534123590216e-05, 'pos_mse': 0.4825380742549896},
 1000: {'loss': 5.1603794418042526e-05, 'pos_mse': 0.5376649498939514},
 1100: {'loss': 5.1371716836001724e-05, 'pos_mse': 0.5104935169219971},
 1200: {'loss': 5.129976125317626e-05, 'pos_mse': 0.5297415852546692},
 1300: {'loss': 5.115093517815694e-05, 'pos_mse': 0.5152726769447327},
 1400: {'loss': 5.0753609684761614e-05, 'pos_mse': 0.5082546472549438},
 1500: {'loss